
# 03 — Data Cleaning

## Project: ThreatTrace

### Objective
Clean the merged network traffic dataset and prepare it for subsequent model development.

### Cleaning Tasks
1. Load the merged dataset generated in Notebook 2.
2. Inspect dataset dimensions, column names, data types, and target labels.
3. Identify missing, infinite, and duplicate values.
4. Handle invalid values and missing data appropriately.
5. Validate the cleaned dataset.
6. Save the cleaned dataset for the next stage.

### Scope
This notebook focuses only on data cleaning. Feature selection, feature engineering, feature scaling, and model training will be performed separately.


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
import gc

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")


Libraries imported successfully.


In [ ]:

# Load the merged dataset from Notebook 2

from pathlib import Path

PROJECT_ROOT = Path.cwd()

DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "cicids2017_merged.csv"

df = pd.read_csv(DATASET_PATH, low_memory=False)

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")

Dataset loaded successfully.
Dataset shape: (2830743, 79)
Total rows: 2,830,743
Total columns: 79


In [5]:

# Inspect the dataset structure

print("First 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.value_counts().rename_axis("Data Type").to_frame("Column Count"))

print("\nDataset memory usage:")
print(f"{df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")


First 5 rows:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.66670,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,33,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,1.100917e+05,18348.62385,109.0,0.0,109,109,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,9174.311927,9174.311927,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,2.307692e+05,38461.53846,52.0,0.0,52,52,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,19230.769230,19230.769230,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,3.529412e+05,58823.52941,34.0,0.0,34,34,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,29411.764710,29411.764710,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,20,0,0,0,0,0,0,1,6,1,6,31,329,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.66670,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,40,0,0,0,0,0,0,2,12,0,0,32,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN



Column names:
[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Coun

,Column Count
Data Type,
int64,54
float64,24
str,1



Dataset memory usage:
1.79 GB


In [6]:

# Remove leading and trailing whitespace from column names

df.columns = df.columns.str.strip()

print("Column names cleaned successfully.")
print("\nFirst 10 column names:")
print(df.columns[:10].tolist())

print("\nTarget column exists:", "Label" in df.columns)


Column names cleaned successfully.

First 10 column names:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']

Target column exists: True


In [7]:

# Inspect data quality before making cleaning decisions

numeric_df = df.select_dtypes(include=[np.number])

missing_total = int(df.isna().sum().sum())
infinite_total = int(np.isinf(numeric_df).sum().sum())
duplicate_total = int(df.duplicated().sum())

print(f"Total rows: {len(df):,}")
print(f"Total columns: {df.shape[1]}")
print(f"Total missing values: {missing_total:,}")
print(f"Total infinite numeric values: {infinite_total:,}")
print(f"Exact duplicate rows: {duplicate_total:,}")

print("\nColumns containing missing values:")
missing_by_column = df.isna().sum()
display(
    missing_by_column[missing_by_column > 0]
    .sort_values(ascending=False)
    .to_frame("Missing Count")
)

print("\nLabels before cleaning:")
display(df["Label"].value_counts(dropna=False).to_frame("Count"))


Total rows: 2,830,743
Total columns: 79
Total missing values: 1,358
Total infinite numeric values: 4,376
Exact duplicate rows: 308,381

Columns containing missing values:


,Missing Count
Flow Bytes/s,1358



Labels before cleaning:


,Count
Label,
BENIGN,2273097
DoS Hulk,231073
PortScan,158930
DDoS,128027
DoS GoldenEye,10293
FTP-Patator,7938
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499


In [8]:

# Create a separate working copy for cleaning
df_clean = df.copy()

# Replace positive and negative infinity in numeric columns with NaN
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

df_clean[numeric_cols] = df_clean[numeric_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

# Count rows affected by missing values before removing them
rows_with_missing = int(df_clean.isna().any(axis=1).sum())
missing_before = int(df_clean.isna().sum().sum())

print(f"Rows containing missing values: {rows_with_missing:,}")
print(f"Total missing values after replacing infinity: {missing_before:,}")

# Remove rows containing missing values
df_clean = df_clean.dropna().copy()

print("\nMissing-value cleaning completed.")
print(f"Rows remaining: {len(df_clean):,}")
print(f"Rows removed: {len(df) - len(df_clean):,}")
print(f"Remaining missing values: {df_clean.isna().sum().sum():,}")


Rows containing missing values: 2,867
Total missing values after replacing infinity: 5,734

Missing-value cleaning completed.
Rows remaining: 2,827,876
Rows removed: 2,867
Remaining missing values: 0


In [9]:

# Examine duplicate records before removing them

feature_cols = [col for col in df_clean.columns if col != "Label"]

# Count exact duplicate rows, including the target label
exact_duplicates = int(df_clean.duplicated().sum())

# Check whether identical feature values have different target labels
label_counts_per_feature_row = (
    df_clean.groupby(feature_cols, dropna=False)["Label"]
    .nunique()
)

conflicting_feature_groups = int(
    (label_counts_per_feature_row > 1).sum()
)

print(f"Exact duplicate rows: {exact_duplicates:,}")
print(
    "Feature groups associated with multiple labels:",
    f"{conflicting_feature_groups:,}"
)


Exact duplicate rows: 307,078
Feature groups associated with multiple labels: 697


In [10]:

# Inspect records where identical features have different labels

feature_cols = [col for col in df_clean.columns if col != "Label"]

label_counts = (
    df_clean.groupby(feature_cols, dropna=False)["Label"]
    .nunique()
)

conflicting_groups = label_counts[label_counts > 1]

print("Conflicting feature groups:", len(conflicting_groups))

# Display label combinations for a sample of conflicting groups
if len(conflicting_groups) > 0:
    conflict_index = conflicting_groups.index[:5]

    conflict_sample = df_clean.set_index(feature_cols).loc[
        conflict_index
    ].reset_index()

    display(
        conflict_sample[
            feature_cols + ["Label"]
        ].sort_values(feature_cols).head(20)
    )
else:
    print("No conflicting feature groups found.")


Conflicting feature groups: 697


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,1,54,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,148148.1481,37037.03704,54.0,0.0,54,54,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,18518.51852,18518.51852,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0,24,0.0,0.0,0,0,0.0,0.0,0,0,PortScan
1,1,54,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,148148.1481,37037.03704,54.0,0.0,54,54,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,18518.51852,18518.51852,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0,24,0.0,0.0,0,0,0.0,0.0,0,0,PortScan
2,1,54,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,148148.1481,37037.03704,54.0,0.0,54,54,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,18518.51852,18518.51852,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0,24,0.0,0.0,0,0,0.0,0.0,0,0,PortScan
3,1,54,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,148148.1481,37037.03704,54.0,0.0,54,54,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,18518.51852,18518.51852,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0,24,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,3,47,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,170212.7660,42553.19149,47.0,0.0,47,47,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,21276.59574,21276.59574,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0,24,0.0,0.0,0,0,0.0,0.0,0,0,PortScan
5,3,47,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,170212.7660,42553.19149,47.0,0.0,47,47,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,21276.59574,21276.59574,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0,24,0.0,0.0,0,0,0.0,0.0,0,0,PortScan
6,3,47,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,170212.7660,42553.19149,47.0,0.0,47,47,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,21276.59574,21276.59574,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0,24,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
7,4,54,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,148148.1481,37037.03704,54.0,0.0,54,54,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,18518.51852,18518.51852,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0,24,0.0,0.0,0,0,0.0,0.0,0,0,PortScan
8,4,54,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,148148.1481,37037.03704,54.0,0.0,54,54,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,18518.51852,18518.51852,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0,24,0.0,0.0,0,0,0.0,0.0,0,0,PortScan
9,4,54,1,1,2,6,2,2,2.0,0.0,6,6,6.0,0.0,148148.1481,37037.03704,54.0,0.0,54,54,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,24,20,18518.51852,18518.51852,2,6,3.333333,2.309401,5.333333,0,0,0,1,0,0,0,0,1,5.0,2.0,6.0,24,0,0,0,0,0,0,1,2,1,6,1024,0,0

In [11]:

# Remove exact duplicate records while retaining the first occurrence

rows_before = len(df_clean)

df_clean = df_clean.drop_duplicates(keep="first").copy()

rows_removed = rows_before - len(df_clean)

print("Duplicate removal completed.")
print(f"Rows before: {rows_before:,}")
print(f"Duplicate rows removed: {rows_removed:,}")
print(f"Rows after: {len(df_clean):,}")
print(f"Remaining exact duplicates: {df_clean.duplicated().sum():,}")


Duplicate removal completed.
Rows before: 2,827,876
Duplicate rows removed: 307,078
Rows after: 2,520,798
Remaining exact duplicates: 0


In [12]:

# Final data-quality checks

numeric_df = df_clean.select_dtypes(include=[np.number])

print("Final dataset shape:", df_clean.shape)
print("Missing values:", int(df_clean.isna().sum().sum()))
print("Infinite numeric values:", int(np.isinf(numeric_df).sum().sum()))
print("Exact duplicate rows:", int(df_clean.duplicated().sum()))
print("Missing target labels:", int(df_clean["Label"].isna().sum()))

print("\nClass distribution:")
display(
    df_clean["Label"]
    .value_counts(dropna=False)
    .rename_axis("Label")
    .to_frame("Count")
)

print("\nData types:")
display(df_clean.dtypes.value_counts().rename_axis("Data Type").to_frame("Count"))


Final dataset shape: (2520798, 79)
Missing values: 0
Infinite numeric values: 0
Exact duplicate rows: 0
Missing target labels: 0

Class distribution:


,Count
Label,
BENIGN,2095057
DoS Hulk,172846
DDoS,128014
PortScan,90694
DoS GoldenEye,10286
FTP-Patator,5931
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3219



Data types:


,Count
Data Type,
int64,54
float64,24
str,1


In [13]:

# Save the cleaned dataset

OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "cicids2017_cleaned.csv"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(OUTPUT_PATH, index=False)

print("Cleaned dataset saved successfully.")
print(f"Output path: {OUTPUT_PATH}")
print(f"Rows: {len(df_clean):,}")
print(f"Columns: {df_clean.shape[1]}")
print(f"File size: {OUTPUT_PATH.stat().st_size / (1024**3):.2f} GB")


Cleaned dataset saved successfully.
Output path: d:\ThreatTrace\data\processed\cicids2017_cleaned.csv
Rows: 2,520,798
Columns: 79
File size: 0.82 GB


In [14]:

# Verify the saved dataset can be read correctly

df_check = pd.read_csv(OUTPUT_PATH, low_memory=False)

numeric_check = df_check.select_dtypes(include=[np.number])

print("Saved dataset verification")
print("-" * 40)
print(f"Shape: {df_check.shape}")
print(f"Missing values: {df_check.isna().sum().sum():,}")
print(f"Infinite numeric values: {np.isinf(numeric_check).sum().sum():,}")
print(f"Duplicate rows: {df_check.duplicated().sum():,}")
print(f"Missing labels: {df_check['Label'].isna().sum():,}")
print(f"Unique classes: {df_check['Label'].nunique()}")

assert df_check.shape == df_clean.shape
assert df_check.isna().sum().sum() == 0
assert np.isinf(numeric_check).sum().sum() == 0
assert df_check.duplicated().sum() == 0
assert df_check["Label"].notna().all()

print("\nAll saved-dataset checks passed.")

Saved dataset verification
----------------------------------------
Shape: (2520798, 79)
Missing values: 0
Infinite numeric values: 0
Duplicate rows: 0
Missing labels: 0
Unique classes: 15

All saved-dataset checks passed.


## Conclusion

The CICIDS2017 merged dataset was cleaned and validated successfully.

### Cleaning operations
- Standardized column names by removing leading and trailing spaces.
- Replaced infinite numeric values with missing values.
- Removed rows containing missing values.
- Removed exact duplicate rows.
- Preserved the target labels and all 15 classes.

### Final dataset
- Rows: 2,520,798
- Columns: 79
- Missing values: 0
- Infinite numeric values: 0
- Exact duplicate rows: 0
- Missing target labels: 0
- Unique classes: 15

The cleaned dataset was saved as `cicids2017_cleaned.csv`.

**Scope note:** Feature engineering, feature scaling, feature selection, class-imbalance handling, and model training are outside the scope of this notebook and will be addressed separately.